top 3 unique ads per top-20 advertiser. picks the highest-spend ad (keyed by creative title) for each of the top advertisers in the v3 advocacy corpus. quick reference for what each of the PAC-style advertisers actually ran during the election window. exports a csv for the report appendix.

setup.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, expr, coalesce, substring, first,
    sum as spark_sum, count as spark_count,
    row_number, desc,
)
from pyspark.sql.window import Window
import pandas as pd

spark = SparkSession.builder \
    .appName('FB_API_top_ads') \
    .config('spark.sql.parquet.output.committer.class',
            'org.apache.parquet.hadoop.ParquetOutputCommitter') \
    .config('mapreduce.fileoutputcommitter.algorithm.version', '2') \
    .getOrCreate()

print('Master:', spark.sparkContext.master)
print('Spark version:', spark.version)

paths.

In [ ]:
V3_PATH       = '/user/s3348393/main/preprocessing/v3/parquet'
TOP_ADS_CSV   = '../data/top_ads_per_advertiser.csv'

TOP_N_ADVERTISERS         = 20
TOP_N_ADS_PER_ADVERTISER  = 3

load v3 advocacy ads (drop candidate / party rows) and find the top 20 advertisers by total spend.

In [ ]:
v3 = (spark.read.parquet(V3_PATH)
        .filter(col('match_type').isNull()))

byline_totals = (v3
    .filter(col('bylines').isNotNull() & col('spend_mid').isNotNull())
    .groupBy('bylines')
    .agg(spark_sum('spend_mid').alias('total_spend'))
    .orderBy(desc('total_spend'))
    .limit(TOP_N_ADVERTISERS)
    .toPandas())

top_bylines = byline_totals['bylines'].tolist()
print(f'Top {TOP_N_ADVERTISERS} advertisers by total spend:')
display(byline_totals)

build a title key per ad. first non-empty element of creative_link_titles, falling back to a truncated creative_body when no link title is set. then group by (byline, title), sum spend, and rank within each byline.

In [ ]:
def first_non_empty(col_name):
    return expr(f"filter({col_name}, x -> x is not null and length(x) > 0)[0]")

# Title key: prefer the link title; fall back to a truncated body so ads
# without a link title still group meaningfully.
ad_title = coalesce(
    first_non_empty('creative_link_titles'),
    substring(first_non_empty('creative_bodies'), 1, 80),
)

ad_spend = (v3
    .filter(col('bylines').isin(top_bylines) & col('spend_mid').isNotNull())
    .withColumn('ad_title', ad_title)
    .filter(col('ad_title').isNotNull())
    .groupBy('bylines', 'ad_title')
    .agg(
        spark_sum('spend_mid').alias('total_spend'),
        spark_count('*').alias('n_runs'),
        first('ad_snapshot_url').alias('example_url'),
    ))

w = Window.partitionBy('bylines').orderBy(desc('total_spend'))
top_ads = (ad_spend
    .withColumn('rank', row_number().over(w))
    .filter(col('rank') <= TOP_N_ADS_PER_ADVERTISER)
    .orderBy('bylines', 'rank')
    .toPandas())

print(f'{len(top_ads)} rows: top {TOP_N_ADS_PER_ADVERTISER} ads for each of the top {TOP_N_ADVERTISERS} advertisers.')

display per advertiser and export. ordering follows total-spend rank.

In [ ]:
# Preserve the advertiser ordering from byline_totals
top_ads['_byline_order'] = top_ads['bylines'].map(
    {b: i for i, b in enumerate(top_bylines)})
top_ads = (top_ads
    .sort_values(['_byline_order', 'rank'])
    .drop(columns='_byline_order')
    .reset_index(drop=True))

# Per-advertiser display
for byline in top_bylines:
    sub = top_ads[top_ads['bylines'] == byline]
    if sub.empty:
        continue
    total = byline_totals.loc[byline_totals['bylines'] == byline, 'total_spend'].iloc[0]
    print(f'\n=== {byline}  (${total:,.0f} total) ===')
    display(sub[['rank', 'ad_title', 'total_spend', 'n_runs']])

# Export to csv for the report appendix
out_cols = ['bylines', 'rank', 'ad_title', 'total_spend', 'n_runs', 'example_url']
top_ads[out_cols].to_csv(TOP_ADS_CSV, index=False)
print(f'\nWrote {TOP_ADS_CSV}')